In [10]:
import sys, os

# Make your project importable
project_root = "/Users/alizedematas/ccm_benchmate"
if project_root not in sys.path:
    sys.path.append(project_root)

from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

from datetime import datetime, timezone

from benchmate.knowledge_base.tables import Project, Structure as StructureTable
from benchmate.knowledge_base.knowledge_base import KnowledgeBase
from benchmate.structure.structure import StructureInfo
from biotite.structure.io.pdb import PDBFile
from biotite.structure import get_chains

# DB connection 
user = "benchmate_db_admin"         
password = "sickkid" 
host = "localhost"
port = 5432
database = "benchmate_db"

# Create sql alchemy engine 
engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}")
SessionLocal = sessionmaker(bind=engine)
session = SessionLocal()

kb = KnowledgeBase(engine)

print("connected; tables:", list(kb.db_tables.keys()))

connected; tables: ['project', 'structure']


In [11]:
# create project

class ProjectWrapper:
    def __init__(self, kb, session, project_id: int):
        self.id = project_id
        self.kb = kb
        self.session = session

project_name = "7Z60_test"

proj = session.query(Project).filter_by(name=project_name).one_or_none()
if proj is None:
    proj = Project(
        name=project_name,
        description="testing StructureInfo tokb/fromkb",
        created_at=datetime.now(timezone.utc),
        updated_at=datetime.now(timezone.utc)
       
    )
    session.add(proj)
    session.commit()
    print("Created new project id:", proj.id)
else:
    print("Reusing existing project id:", proj.id)

project = ProjectWrapper(kb, session, proj.id)

Created new project id: 11


In [12]:
# Read PDB file 

pdb_path = "7Z60.pdb"   
assert os.path.exists(pdb_path), f"PDB not found at {pdb_path}"

pdb = PDBFile.read(pdb_path)
atoms = pdb.get_structure(model=1)
chains = list(get_chains(atoms))

# Create structureInfo 

info = StructureInfo(
    name="6PYA_test",
    atoms=atoms,
    chains=chains,
    annotations={"source": "plaintext test"},
)

print("Prepared StructureInfo:")
print("name:", info.name)
print("chains:", info.chains)
print("atom count:", info.atoms.array_length())


# store in database 
inserted_id = info.to_kb(project)
print("Inserted structure id:", inserted_id)

# 4) Load back from DB
info_back = StructureInfo.from_kb(project, inserted_id)

print("Roundtrip name:", info_back.name)
print("Roundtrip chains:", info_back.chains)
print("Roundtrip atoms length:", info_back.atoms.array_length())
print("Roundtrip annotations:", info_back.annotations)

# consistency checks 
assert info_back.name == info.name
assert info_back.chains == info.chains
assert info_back.annotations == info.annotations
assert info_back.atoms.array_length() == info.atoms.array_length()

print("StructureInfo to_kb/from_kb roundtrip OK :)")

Prepared StructureInfo:
name: 6PYA_test
chains: ['A', 'B', 'A', 'B', 'A', 'B']
atom count: 2021
Inserted structure id: 35
Roundtrip name: 6PYA_test
Roundtrip chains: ['A', 'B', 'A', 'B', 'A', 'B']
Roundtrip atoms length: 2021
Roundtrip annotations: {'source': 'plaintext test'}
StructureInfo to_kb/from_kb roundtrip OK :)
